# T2 — SPARQL Generator Training

Fine-tunes `flan-t5-xl` with LoRA on combined DBpedia + Wikidata.

Input: `generate sparql [DBpedia/Wikidata]: question + plan + schema`  
Output: normalized SPARQL query

## 1. Install dependencies

In [1]:
!pip install -q transformers==4.41.2 peft==0.10.0 accelerate==0.29.3 datasets evaluate sentencepiece

## 2. GPU check

In [2]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB')

CUDA available : True
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM: 102.0 GB


## 3. Paths

In [3]:
import os
import json
import zipfile

!pip install -q gdown
import gdown

DATA_DIR       = './'
CHECKPOINT_DIR = './t2_checkpoints'
MODEL_NAME     = 'google/flan-t5-xl'

os.makedirs(DATA_DIR,       exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Download train + val from Google Drive ────────────────────
files = {
    't2_train.json': '1jv3XmPMVGvO3hMnhppwHZ_R033uGkZB0',
    't2_val.json'  : '1ZG19Y1AfsrcoAvAdM9phfhhstas_GBOY',
}

for fname, file_id in files.items():
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        print(f'Downloading {fname}...')
        gdown.download(f'https://drive.google.com/uc?id={file_id}', path, quiet=False)
    else:
        print(f'{fname} already exists')

TRAIN_PATH = os.path.join(DATA_DIR, 't2_train.json')
VAL_PATH   = os.path.join(DATA_DIR, 't2_val.json')

print(f'\nTrain: {TRAIN_PATH}')
print(f'Val  : {VAL_PATH}')

Downloading...
From: https://drive.google.com/uc?id=1jv3XmPMVGvO3hMnhppwHZ_R033uGkZB0
To: /teamspace/studios/this_studio/t2_train.json
100%|██████████| 20.5M/20.5M [00:00<00:00, 59.0MB/s]


Downloading...
From: https://drive.google.com/uc?id=1ZG19Y1AfsrcoAvAdM9phfhhstas_GBOY
To: /teamspace/studios/this_studio/t2_val.json
100%|██████████| 1.95M/1.95M [00:00<00:00, 170MB/s]


Train: ./t2_train.json
Val  : ./t2_val.json


## 4. Load dataset

In [4]:
with open(TRAIN_PATH) as f:
    train_data = json.load(f)
with open(VAL_PATH) as f:
    val_data = json.load(f)

dbp_train = sum(1 for x in train_data if '[DBpedia]'  in x['input'])
wik_train = sum(1 for x in train_data if '[Wikidata]' in x['input'])
dbp_val   = sum(1 for x in val_data   if '[DBpedia]'  in x['input'])
wik_val   = sum(1 for x in val_data   if '[Wikidata]' in x['input'])

print(f'Train: {len(train_data)}  (DBpedia: {dbp_train} | Wikidata: {wik_train})')
print(f'Val  : {len(val_data)}   (DBpedia: {dbp_val}   | Wikidata: {wik_val})')
print(f'\nSample input :\n{train_data[0]["input"]}')
print(f'\nSample output:\n{train_data[0]["output"]}')

Train: 24244  (DBpedia: 6878 | Wikidata: 17366)
Val  : 2311   (DBpedia: 382   | Wikidata: 1929)

Sample input :
generate sparql [Wikidata]: Which was the wife of Erich Honecker in the series ordinal 3?
plan: step1: action: find_entity | surface_form: Erich Honecker | output_variable: ?erichhonecker | semantic_type: ENTITY || step2: action: find_statement | property: spouse | subject_variable: ?erichhonecker | output_variable: ?s | semantic_type: PROPERTY || step3: action: find_object | property: series ordinal | subject_variable: ?s | output_variable: ?x | semantic_type: QUALIFIER | is_qualifier: True || step4: action: find_object | property: spouse | subject_variable: ?s | output_variable: ?obj | semantic_type: PROPERTY || step5: action: filter | filter_variable: ?x | operator: contains | value: 3 | value_type: string | semantic_type: LITERAL
schema:
  Erich Honecker -> wd:Q2607
  spouse -> p:P26
  series ordinal -> pq:P1545

Sample output:
SELECT ?obj WHERE { wd:Q2607 p:P26 ?s . ?s p

## 5. Load tokenizer

In [5]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size}')

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Vocab size: 32000


## 6. Pre-tokenize

`max_input=256` (p99 input = 157 words ~210 tokens)  
`max_target=128` (p99 output = 28 words ~40 tokens)

In [6]:
from datasets import Dataset as HFDataset

MAX_INPUT  = 256
MAX_TARGET = 128

def preprocess(examples):
    inputs = tokenizer(
        examples['input'],
        max_length  = MAX_INPUT,
        padding     = 'max_length',
        truncation  = True,
    )
    targets = tokenizer(
        examples['output'],
        max_length  = MAX_TARGET,
        padding     = 'max_length',
        truncation  = True,
    )
    labels = [
        [-100 if t == tokenizer.pad_token_id else t for t in l]
        for l in targets['input_ids']
    ]
    inputs['labels'] = labels
    return inputs

print('Pre-tokenizing train...')
hf_train = HFDataset.from_list(train_data).map(preprocess, batched=True, batch_size=1000)
print('Pre-tokenizing val...')
hf_val   = HFDataset.from_list(val_data).map(preprocess,   batched=True, batch_size=1000)
hf_train.set_format('torch')
hf_val.set_format('torch')

print(f'Done! Train: {len(hf_train)} | Val: {len(hf_val)}')

Pre-tokenizing train...


Map:   0%|          | 0/24244 [00:00<?, ? examples/s]

Pre-tokenizing val...


Map:   0%|          | 0/2311 [00:00<?, ? examples/s]

Done! Train: 24244 | Val: 2311


## 7. Metrics

In [7]:
from collections import Counter
import numpy as np

def token_f1(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    common   = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall    = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    exact = [p.strip() == g.strip() for p, g in zip(decoded_preds, decoded_labels)]
    tf1   = [token_f1(p.strip(), g.strip()) for p, g in zip(decoded_preds, decoded_labels)]

    # per-KG token F1
    dbp_tf1, wik_tf1 = [], []
    for i, item in enumerate(val_data):
        score = tf1[i]
        if '[DBpedia]' in item['input']:
            dbp_tf1.append(score)
        else:
            wik_tf1.append(score)

    result = {
        'exact_match'      : round(np.mean(exact)   * 100, 2),
        'token_f1'         : round(np.mean(tf1)      * 100, 2),
    }
    if dbp_tf1:
        result['token_f1_dbpedia']  = round(np.mean(dbp_tf1) * 100, 2)
    if wik_tf1:
        result['token_f1_wikidata'] = round(np.mean(wik_tf1) * 100, 2)
    return result

print('Metrics defined: exact_match, token_f1, token_f1_dbpedia, token_f1_wikidata')

Metrics defined: exact_match, token_f1, token_f1_dbpedia, token_f1_wikidata


## 8. Training arguments

In [16]:
from transformers import Seq2SeqTrainingArguments
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

training_args = Seq2SeqTrainingArguments(
    output_dir                  = CHECKPOINT_DIR,
    num_train_epochs            = 10,
    learning_rate               = 5e-4,

    per_device_train_batch_size = 32,
    gradient_accumulation_steps = 2,               # 32 x 2 = 64 effective batch
    per_device_eval_batch_size  = 32,

    bf16                        = True,
    tf32                        = True,

    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    weight_decay                = 0.01,

    evaluation_strategy         = 'steps',
    save_strategy               = 'steps',
    eval_steps                  = 758,          # ~2 epochs (24k/32 = 756 steps/epoch x2)
    save_steps                  = 758,

    load_best_model_at_end      = True,
    metric_for_best_model       = 'token_f1',
    greater_is_better           = True,
    predict_with_generate       = True,
    generation_max_length       = 128,

    logging_steps               = 20,
    save_total_limit            = 2,
    dataloader_num_workers      = 4,
    dataloader_pin_memory       = True,
    group_by_length             = False,
)

print('Training args ready')
print(f'Effective batch size : {32 * 2}')
print(f'Steps per epoch      : {len(train_data) // 32}')
print(f'Eval every           : 3800 steps (~2 epochs)')
print(f'Checkpoints dir      : {CHECKPOINT_DIR}')

Training args ready
Effective batch size : 64
Steps per epoch      : 757
Eval every           : 3800 steps (~2 epochs)
Checkpoints dir      : ./t2_checkpoints


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## 9. Build trainer

In [17]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback, default_data_collator

def build_trainer(model, tokenizer):
    return Seq2SeqTrainer(
        model           = model,
        args            = training_args,
        train_dataset   = hf_train,
        eval_dataset    = hf_val,
        tokenizer       = tokenizer,
        data_collator   = default_data_collator,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
    )

print('build_trainer() ready')

build_trainer() ready


---
## 10-A. 🟢 FRESH START

> Skip to 10-B if resuming.

In [18]:
from transformers import T5ForConditionalGeneration
from peft import get_peft_model, LoraConfig, TaskType
import torch

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.bfloat16,
    device_map  = 'auto'
)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ['q', 'k', 'v', 'o'],
    bias           = 'none'
)

model  = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainer = build_trainer(model, tokenizer)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,874,368 || all params: 2,868,631,552 || trainable%: 0.6579572056523207


## 10-B. 🔄 RESUME

> Skip to 10-A if starting fresh.

In [ ]:
import os, torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from peft import PeftModel

all_ckpts = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')],
    key=lambda x: int(x.split('-')[1])
)

if not all_ckpts:
    raise RuntimeError(f'No checkpoints in {CHECKPOINT_DIR}. Use 10-A.')

LATEST_CKPT = os.path.join(CHECKPOINT_DIR, all_ckpts[-1])
print(f'Resuming from: {LATEST_CKPT}')

base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.bfloat16,
    device_map  = 'auto'
)
model     = PeftModel.from_pretrained(base_model, LATEST_CKPT, is_trainable=True)
tokenizer = T5Tokenizer.from_pretrained(LATEST_CKPT)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
model.train()

trainer = build_trainer(model, tokenizer)
print('Model loaded — ready to resume')

---
## 11-A. 🟢 FRESH START — Train

In [19]:
import time

start = time.time()
trainer.train()
elapsed = (time.time() - start) / 3600
print(f'\nDone in {elapsed:.2f} hours')

Step,Training Loss,Validation Loss,Exact Match,Token F1,Token F1 Dbpedia,Token F1 Wikidata
758,0.126500,0.126677,65.770000,89.460000,95.060000,88.350000
1516,0.107200,0.114004,67.030000,91.310000,95.770000,90.430000
2274,0.097900,0.109217,68.500000,92.060000,96.790000,91.130000
3032,0.096000,0.107661,68.540000,92.110000,96.730000,91.200000


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/hug

KeyboardInterrupt: 

## 11-B. 🔄 RESUME — Train

In [ ]:
import time

print(f'Resuming from {LATEST_CKPT}...')
start = time.time()
trainer.train(resume_from_checkpoint=LATEST_CKPT)
elapsed = (time.time() - start) / 3600
print(f'\nDone in {elapsed:.2f} hours')

## 12. Final evaluation

In [ ]:
results = trainer.evaluate()

print('\n' + '='*55)
print('FINAL RESULTS — T2 SPARQL Generator')
print('='*55)
print(f"Exact Match        : {results.get('eval_exact_match',      'N/A')}%")
print(f"Token F1 (all)     : {results.get('eval_token_f1',         'N/A')}%")
print(f"Token F1 DBpedia   : {results.get('eval_token_f1_dbpedia', 'N/A')}%")
print(f"Token F1 Wikidata  : {results.get('eval_token_f1_wikidata','N/A')}%")
print(f"Eval Loss          : {results.get('eval_loss',             'N/A')}")

## 13. Save LoRA adapter

In [ ]:
import shutil

SAVE_PATH = './t2_lora_adapter'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Saved to {SAVE_PATH}')

shutil.make_archive('t2_lora_adapter', 'zip', SAVE_PATH)
print('Zipped.')

## 14. Inference — quick test

In [21]:
import torch, re

model.eval()

def fix_sparql(sparql):
    sparql = sparql.strip()
    sparql = re.sub(r'SELECT\s*\?', 'SELECT ?', sparql)
    sparql = re.sub(r'DISTINCT\s*\?', 'DISTINCT ?', sparql)
    sparql = re.sub(r'ASK\s*\{', 'ASK {', sparql)
    sparql = re.sub(r'\.\s*\?', ' . ?', sparql)  
    sparql = re.sub(r'\.\s*\}', ' }', sparql)
    if 'WHERE' in sparql and '{' not in sparql:
        sparql = re.sub(r'(WHERE)\s+', r'\1 { ', sparql)
        sparql = sparql.rstrip() + ' }'
    return sparql

def predict(input_text):
    inputs = tokenizer(
        input_text,
        return_tensors = 'pt',
        max_length     = 256,
        truncation     = True
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length     = 128,
            num_beams      = 4,
            early_stopping = True
        )
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return fix_sparql(raw)

# test one DBpedia and one Wikidata from val
dbp_sample = next(x for x in val_data if '[DBpedia]'  in x['input'])
wik_sample = next(x for x in val_data if '[Wikidata]' in x['input'])

for sample in [dbp_sample, wik_sample]:
    predicted = predict(sample['input'])
    exact     = predicted.strip() == sample['output'].strip()
    print('INPUT:')
    print(sample['input'])
    print('\nPREDICTED:')
    print(predicted)
    print('\nEXPECTED:')
    print(sample['output'])
   
    print('-'*60)

INPUT:
generate sparql [DBpedia]: Which university has affiliations to Graham Holdings and Kaplan, Inc.?
plan: step1: action: find_entity | surface_form: Graham Holdings Company | output_variable: ?graham_holdings_comp | semantic_type: ENTITY || step2: action: find_entity | surface_form: Kaplan, Inc. | output_variable: ?kaplan_inc | semantic_type: ENTITY || step3: action: find_subjects | property: affiliations | object_variable: ?graham_holdings_comp | output_variable: ?uri | semantic_type: PROPERTY || step4: action: find_subjects | property: affiliations | object_variable: ?kaplan_inc | semantic_type: PROPERTY | join_type: intersect || step5: action: filter_type | type: University | filter_variable: ?uri | semantic_type: CLASS || step6: action: distinct | semantic_type: MODIFIER
schema:
  Graham Holdings Company -> dbr:Graham_Holdings_Company
  Kaplan, Inc. -> dbr:Kaplan,_Inc.
  affiliations -> dbp:affiliations
  University -> dbo:University

PREDICTED:
SELECT DISTINCT ?uri WHERE { ?u